# MSCI EUR SMALL：逐变量证据驱动的 family composite 与 incremental 分析

本 notebook 只负责搭建可复现 pipeline，不在提交时执行。它把逐变量证据报告中每个 family 的 long、cycle、short 三条腿各取一条，按组件等权构建新的 family 主综合因子；随后用 screen 中的原有 family score 做基准，并逐条测试单变量加入原有 family 的边际效果。

所有结果写入 exports/factor_family_pipeline_SMALL。短周期的 robust_score 只作诊断，不与完整历史的 robust_score 做绝对比较。

In [ ]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "func.py").exists():
    raise RuntimeError(
        "请把 Jupyter 的工作目录设为 C:\\dev\\factor_backtest 后再运行本 notebook。"
    )

from func import (
    calculate_benchmark_performance,
    calculate_performance_ratios,
    combine_backtest_performances,
    export_backtest_results,
    load_backtest_data,
    test_composite_signals,
    test_incremental_signals,
)
from factor_config import signal_options

print("研究函数已加载。")


In [ ]:
MARKET = "EUROPE SMALL CAP"
BENCHMARK = "MSCI EUR SMALL"
START_DATE = "2007-12-01"
PERCENTILE = 0.13
N_JOBS = 1
PERIOD_BREAKPOINTS = [2009, 2013, 2017, 2020, 2022, 2024, 2026]
OUTPUT_NAME = "factor_family_pipeline_SMALL"
EVIDENCE_REPORT = Path(
    r"C:\dev\factor_backtest\exports\_agent_work\small_regime_fullpool.md"
)

BASELINE_CANDIDATES = {
    "growth": ("GROWTH_SCORE_FS_SECTOR", "Growth Avg Percentile"),
    "quality": ("Quality Avg Percentile", "MARGIN_SCORE_FS_SECTOR"),
    "momentum": ("MOMENTUM_SCORE_FS_SECTOR", "Mom Avg Percentile"),
    "value": ("VALUE_SCORE_FS_SECTOR", "Value Avg Percentile"),
    "dividend": ("Dividend Avg Percentile", "Dividend_NTM Avg Percentile"),
}
SELECTIONS = {
    "dividend": [
        {"role": "long", "variable": "PCT DvdYield FY1", "dimension": "level",
         "higher_is_better": True, "evidence_class": "long_core",
         "evidence_note": "跨期稳定的 FY1 dividend-yield level"},
        {"role": "cycle", "variable": "DVD Yield FY0", "dimension": "rank_diff_6",
         "higher_is_better": True, "evidence_class": "cycle",
         "evidence_note": "报告中的长窗口 yield rank improvement"},
        {"role": "short", "variable": "DVD Yield FY1", "dimension": "pct_1",
         "higher_is_better": True, "evidence_class": "short_tactical",
         "evidence_note": "短窗口 FY1 yield change，作为战术腿"},
    ],
    "growth": [
        {"role": "long", "variable": "PCT Hist GrossInc", "dimension": "rank_diff_3",
         "higher_is_better": True, "evidence_class": "long_core",
         "evidence_note": "历史 gross income trend 的跨期候选"},
        {"role": "cycle", "variable": "PCT Hist Sales", "dimension": "rank_diff_6",
         "higher_is_better": True, "evidence_class": "cycle",
         "evidence_note": "历史 sales 的较慢变化腿"},
        {"role": "short", "variable": "Revenue 5Y CAGR", "dimension": "pct_1",
         "higher_is_better": True, "evidence_class": "short_tactical",
         "evidence_note": "近期报告中表现强但覆盖较窄的 revenue change"},
    ],
    "momentum": [
        {"role": "long", "variable": "SP Price Target CIQ", "dimension": "pct_6",
         "higher_is_better": True, "evidence_class": "long_core",
         "evidence_note": "价格目标的中长期变化"},
        {"role": "cycle", "variable": "SP Price Close CIQ", "dimension": "pct_12",
         "higher_is_better": True, "evidence_class": "cycle",
         "evidence_note": "较慢价格趋势"},
        {"role": "short", "variable": "SP Price Target CIQ", "dimension": "pct_1",
         "higher_is_better": True, "evidence_class": "short_tactical",
         "evidence_note": "短窗口 price-target revision"},
    ],
    "quality": [
        {"role": "long", "variable": "PCT ROE", "dimension": "diff_3",
         "higher_is_better": True, "evidence_class": "long_core",
         "evidence_note": "ROE improvement 的跨期稳定证据"},
        {"role": "cycle", "variable": "Net Debt", "dimension": "diff_6",
         "higher_is_better": False, "evidence_class": "cycle",
         "evidence_note": "杠杆下降；lower-is-better 方向显式设置"},
        {"role": "short", "variable": "Net Debt to Ebit", "dimension": "diff_1",
         "higher_is_better": False, "evidence_class": "short_tactical",
         "evidence_note": "近期去杠杆变化，先作为 incremental candidate"},
    ],
    "value": [
        {"role": "long", "variable": "Earns Yield FY0", "dimension": "level",
         "higher_is_better": True, "evidence_class": "long_core",
         "evidence_note": "跨期收益率 level"},
        {"role": "cycle", "variable": "EV To EBITDA LTM", "dimension": "pct_3",
         "higher_is_better": False, "evidence_class": "cycle",
         "evidence_note": "估值倍数下降；lower-is-better 方向显式设置"},
        {"role": "short", "variable": "Earns Yield FY1", "dimension": "diff_1",
         "higher_is_better": True, "evidence_class": "short_tactical",
         "evidence_note": "近期 forward earnings-yield improvement"},
    ],
}


## 1. 选择清单与等权规则

每个 family 固定三条组件：long、cycle、short；每条组件权重为 1.0。最终 family score 在组件贡献相加后，再按既有 neutral rank 逻辑转为可回测的 cross-sectional score。若一个 raw variable 出现两次但 horizon 不同，两个 dimension 仍分别贡献一个等权组件。

In [ ]:

def make_single_config(spec):
    kwargs = {"higher_is_better": bool(spec["higher_is_better"])}
    kwargs[spec["dimension"]] = 1.0
    return {spec["variable"]: signal_options(**kwargs)}


def make_family_config(specs):
    config = {}
    for spec in specs:
        variable = spec["variable"]
        if variable not in config:
            config[variable] = signal_options(
                higher_is_better=bool(spec["higher_is_better"])
            )
        config[variable][f"weight_{spec['dimension']}"] = 1.0
    return config


def make_baseline_config(family):
    variable = BASELINE_COLUMNS[family]
    return {variable: signal_options(level=1.0, higher_is_better=True)}


SELECTION_ROWS = []
for family, specs in SELECTIONS.items():
    for index, spec in enumerate(specs, start=1):
        SELECTION_ROWS.append(
            {
                "market": MARKET,
                "family": family,
                "component_index": index,
                "role": spec["role"],
                "variable": spec["variable"],
                "dimension": spec["dimension"],
                "higher_is_better": spec["higher_is_better"],
                "evidence_class": spec["evidence_class"],
                "evidence_note": spec["evidence_note"],
                "source_report": str(EVIDENCE_REPORT),
                "composite_weight": 1.0,
                "baseline_column": None,
            }
        )
SELECTION_MANIFEST = pd.DataFrame(SELECTION_ROWS)
display(SELECTION_MANIFEST)


In [ ]:

DATA_DIR = REPO_ROOT / "data"
SCREEN_PATH = DATA_DIR / "screen_aggregate.parquet"
RETURNS_PATH = DATA_DIR / "returns.parquet"
EXPORT_ROOT = REPO_ROOT / "exports"
EXPORT_DIR = EXPORT_ROOT / OUTPUT_NAME
LIST_NOIRE_PATH = None

try:
    import pyarrow.parquet as pq
    available_columns = set(pq.ParquetFile(SCREEN_PATH).schema_arrow.names)
except Exception as error:
    raise RuntimeError("读取 screen parquet schema 失败，请确认 pyarrow 可用。") from error

BASELINE_COLUMNS = {}
for family, candidates in BASELINE_CANDIDATES.items():
    selected = next((candidate for candidate in candidates if candidate in available_columns), None)
    if selected is None:
        raise KeyError(f"找不到 {family} 的原有 screen 因子列：{candidates}")
    BASELINE_COLUMNS[family] = selected

if "SELECTION_MANIFEST" in globals():
    SELECTION_MANIFEST["baseline_column"] = SELECTION_MANIFEST["family"].map(BASELINE_COLUMNS)

SELECTED_RAW_VARIABLES = sorted(
    {
        spec["variable"]
        for specs in SELECTIONS.values()
        for spec in specs
    }
)
LOAD_VARIABLES = list(
    dict.fromkeys(SELECTED_RAW_VARIABLES + list(BASELINE_COLUMNS.values()))
)

screen, returns = load_backtest_data(
    screen_path=SCREEN_PATH,
    returns_path=RETURNS_PATH,
    variables=LOAD_VARIABLES,
    bench=BENCHMARK,
    start_date=START_DATE,
    lookback_periods=12,
    compact_dtypes=True,
)
screen["Date"] = pd.to_datetime(screen["Date"])

missing = [column for column in LOAD_VARIABLES if column not in screen.columns]
if missing:
    raise KeyError(f"加载后缺失变量：{missing}")
if f"Weight in {BENCHMARK}" not in screen.columns:
    raise KeyError(f"screen 中没有 Weight in {BENCHMARK}")

BENCH_PERF = calculate_benchmark_performance(
    screen=screen,
    returns=returns,
    bench=BENCHMARK,
    start_date=START_DATE,
)

MONTHLY_BASE_CACHE = {}
RUN_OPTIONS = {
    "bench": BENCHMARK,
    "bench_perf": BENCH_PERF,
    "percentile": PERCENTILE,
    "start_date": START_DATE,
    "freq_rebal": 1,
    "fill_method": "copy",
    "n_jobs": N_JOBS,
    "retain_builders": False,
    "monthly_base_cache": MONTHLY_BASE_CACHE,
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "show_plot": False,
    "build_figure": False,
}

print(f"screen={screen.shape}, returns={returns.shape}")
print(f"benchmark={BENCHMARK}; baseline columns={BASELINE_COLUMNS}")


## 2. 构建并回测新的 family composite 与 screen 原有 factor

In [ ]:

COMPOSITE_CONFIGS = {}
for family, specs in SELECTIONS.items():
    COMPOSITE_CONFIGS[f"new_family_{family}"] = make_family_config(specs)
for family in SELECTIONS:
    COMPOSITE_CONFIGS[f"screen_baseline_{family}"] = make_baseline_config(family)

composite_batch = test_composite_signals(
    screen=screen,
    returns=returns,
    composite_configs=COMPOSITE_CONFIGS,
    list_noire_path=LIST_NOIRE_PATH,
    score_prefix="Score_FamilyPipeline",
    **RUN_OPTIONS,
)
screen = composite_batch["screen"]
print("已完成新 family composite 与 screen 原有 factor 的对照回测。")


## 3. 对每个单变量做 incremental analysis

每个 batch 都重新以同一 family 的 screen baseline 为基准，只加入一条变量×dimension 组件。这样输出的 delta_active_cagr、delta_top_worst_cagr、delta_top_information_ratio、delta_robust_score、delta_active_max_drawdown 和 delta_tracking_error_annualized 可以直接用于判断边际贡献。

In [ ]:

incremental_batches = {}
for family, specs in SELECTIONS.items():
    for index, spec in enumerate(specs, start=1):
        batch_key = f"{family}__{spec['role']}__{index}"
        incremental_batches[batch_key] = test_incremental_signals(
            screen=screen,
            returns=returns,
            baseline_config=make_baseline_config(family),
            candidate_config=make_single_config(spec),
            list_noire_path=LIST_NOIRE_PATH,
            **RUN_OPTIONS,
        )
        screen = incremental_batches[batch_key]["screen"]

all_results = {
    "composite_comparison": composite_batch,
    "incremental": incremental_batches,
}
print(f"已完成 {len(incremental_batches)} 个单腿 incremental batch。")


## 4. 规范化导出

下面的导出同时保留 official metrics、performance curve、family composite 对照表、逐单腿 incremental 表和 selection manifest。不要只看 total CAGR；优先按 completed economic periods、Top/Worst、IR、drawdown、tracking error 和 gate 一起筛选。

In [ ]:

exported = export_backtest_results(
    results=all_results,
    output_dir=EXPORT_ROOT,
    export_name=OUTPUT_NAME,
    export_html=False,
    export_png=False,
    export_holdings=False,
)
EXPORT_DIR = Path(exported["export_dir"])

metrics = pd.read_csv(EXPORT_DIR / "backtest_metrics.csv")
with (EXPORT_DIR / "backtest_registry.json").open("r", encoding="utf-8") as handle:
    registry = json.load(handle)
path_by_name = {
    entry.get("metadata", {}).get("test_name"): entry.get("test_path")
    for entry in registry
    if entry.get("metadata", {}).get("test_name") and entry.get("test_path")
}

METRIC_COLUMNS = [
    "active_cagr",
    "top_worst_cagr",
    "top_information_ratio",
    "robust_score",
    "active_max_drawdown",
    "tracking_error_annualized",
    "min_rolling_3y_cagr",
    "top_bench_ratio",
    "top_worst_ratio",
    "top_annualized_return",
    "bench_annualized_return",
    "observation_count",
    "years",
]
COMPARABILITY_COLUMNS = [
    "robust_score_comparable"
] if "robust_score_comparable" in metrics.columns else []


def _path_for(test_name):
    if test_name not in path_by_name:
        raise KeyError(f"导出 registry 中找不到 test_name={test_name}")
    return path_by_name[test_name]


def _metric_slice(test_path):
    return metrics.loc[metrics["test_path"].eq(test_path)].copy()


family_comparison_parts = []
family_names = list(SELECTIONS)
for family in family_names:
    new_rows = _metric_slice(_path_for(f"new_family_{family}"))
    base_rows = _metric_slice(_path_for(f"screen_baseline_{family}"))
    left_columns = ["period_id", "scope", "period_label", *METRIC_COLUMNS, *COMPARABILITY_COLUMNS]
    right_columns = ["period_id", "scope", *METRIC_COLUMNS, *COMPARABILITY_COLUMNS]
    left = new_rows[left_columns].rename(
        columns={column: f"{column}_new" for column in [*METRIC_COLUMNS, *COMPARABILITY_COLUMNS]}
    )
    right = base_rows[right_columns].rename(
        columns={column: f"{column}_screen" for column in [*METRIC_COLUMNS, *COMPARABILITY_COLUMNS]}
    )
    joined = left.merge(right, on=["period_id", "scope"], how="outer")
    joined.insert(0, "family", family)
    for column in METRIC_COLUMNS:
        joined[f"delta_{column}"] = (
            joined[f"{column}_new"] - joined[f"{column}_screen"]
        )
    joined["new_perf_gate"] = (
        joined["active_cagr_new"].gt(0)
        & joined["top_worst_cagr_new"].gt(0)
        & joined["top_information_ratio_new"].gt(0)
    )
    joined["screen_perf_gate"] = (
        joined["active_cagr_screen"].gt(0)
        & joined["top_worst_cagr_screen"].gt(0)
        & joined["top_information_ratio_screen"].gt(0)
    )
    joined["performance_improved"] = (
        joined["delta_active_cagr"].gt(0)
        & joined["delta_top_worst_cagr"].gt(0)
        & joined["delta_top_information_ratio"].gt(0)
    )
    joined["risk_not_worse"] = (
        joined["delta_active_max_drawdown"].le(0)
        & joined["delta_tracking_error_annualized"].le(0)
    )
    if COMPARABILITY_COLUMNS:
        comparable_period = (
            joined["scope"].eq("total")
            | (
                joined["robust_score_comparable_new"].astype(str).str.lower().isin(["true", "1", "yes"])
                & joined["robust_score_comparable_screen"].astype(str).str.lower().isin(["true", "1", "yes"])
            )
        )
    else:
        comparable_period = joined["scope"].eq("total")
    joined["strict_comparable_improvement"] = (
        comparable_period
        & joined["performance_improved"]
        & joined["risk_not_worse"]
        & joined["robust_score_new"].gt(joined["robust_score_screen"])
    )
    family_comparison_parts.append(joined)

family_comparison = pd.concat(family_comparison_parts, ignore_index=True)
family_comparison.to_csv(
    EXPORT_DIR / "family_composite_vs_screen.csv", index=False
)
family_comparison.loc[family_comparison["period_id"].eq("total")].to_csv(
    EXPORT_DIR / "family_composite_vs_screen_total.csv", index=False
)

incremental_lookup = {}
for family, specs in SELECTIONS.items():
    for index, spec in enumerate(specs, start=1):
        incremental_lookup[f"{family}__{spec['role']}__{index}"] = {
            "family": family,
            "role": spec["role"],
            "variable": spec["variable"],
            "dimension": spec["dimension"],
            "higher_is_better": spec["higher_is_better"],
        }

incremental_rows = []
for test_group in metrics.loc[
    metrics["test_type"].isin(
        ["incremental_baseline", "incremental_candidate"]
    ),
    "test_group",
].dropna().unique():
    group_rows = metrics.loc[metrics["test_group"].eq(test_group)].copy()
    baseline_rows = group_rows.loc[
        group_rows["test_type"].eq("incremental_baseline")
    ]
    candidate_rows = group_rows.loc[
        group_rows["test_type"].eq("incremental_candidate")
    ]
    batch_key = str(test_group).split(" / ")[-1]
    spec_info = incremental_lookup.get(batch_key, {})
    for _, candidate in candidate_rows.iterrows():
        baseline = baseline_rows.loc[
            baseline_rows["period_id"].eq(candidate["period_id"])
        ]
        if baseline.empty:
            continue
        baseline = baseline.iloc[0]
        row = {
            "batch_key": batch_key,
            **spec_info,
            "period_id": candidate["period_id"],
            "scope": candidate["scope"],
            "period_label": candidate.get("period_label"),
            "candidate_test_path": candidate["test_path"],
            "baseline_test_path": baseline["test_path"],
        }
        for column in METRIC_COLUMNS:
            row[f"{column}_candidate"] = candidate.get(column)
            row[f"{column}_baseline"] = baseline.get(column)
            row[f"delta_{column}"] = (
                candidate.get(column) - baseline.get(column)
            )
        row["candidate_perf_gate"] = (
            candidate["active_cagr"] > 0
            and candidate["top_worst_cagr"] > 0
            and candidate["top_information_ratio"] > 0
        )
        row["baseline_perf_gate"] = (
            baseline["active_cagr"] > 0
            and baseline["top_worst_cagr"] > 0
            and baseline["top_information_ratio"] > 0
        )
        row["incremental_perf_improved"] = (
            row["delta_active_cagr"] > 0
            and row["delta_top_worst_cagr"] > 0
            and row["delta_top_information_ratio"] > 0
        )
        row["incremental_risk_not_worse"] = (
            row["delta_active_max_drawdown"] <= 0
            and row["delta_tracking_error_annualized"] <= 0
        )
        incremental_rows.append(row)

incremental_effects = pd.DataFrame(incremental_rows)
incremental_effects.to_csv(EXPORT_DIR / "incremental_effects.csv", index=False)
incremental_effects.loc[
    incremental_effects["period_id"].eq("total")
].to_csv(EXPORT_DIR / "incremental_effects_total.csv", index=False)

performance_selections = {}
first_baseline_path = None
for family in family_names:
    new_path = _path_for(f"new_family_{family}")
    base_path = _path_for(f"screen_baseline_{family}")
    performance_selections[f"{family}_new"] = (new_path, "Top")
    performance_selections[f"{family}_screen"] = (base_path, "Top")
    first_baseline_path = first_baseline_path or base_path
performance_selections["Benchmark"] = (first_baseline_path, "Bench")

top_curves = combine_backtest_performances(
    export_dir=EXPORT_DIR,
    selections=performance_selections,
)
top_curves.to_csv(EXPORT_DIR / "performance_top_curves.csv", index=True)
top_ratios = calculate_performance_ratios(top_curves, benchmark_column="Benchmark")
top_ratios.to_csv(EXPORT_DIR / "performance_ratios.csv", index=True)

run_manifest = {
    "market": MARKET,
    "benchmark": BENCHMARK,
    "start_date": START_DATE,
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "percentile": PERCENTILE,
    "rebalancing_frequency": 1,
    "fill_method": "copy",
    "baseline_columns": BASELINE_COLUMNS,
    "selection_manifest": SELECTION_ROWS,
    "source_evidence_report": str(EVIDENCE_REPORT),
    "gate": {
        "performance": "active_cagr > 0 and top_worst_cagr > 0 and top_information_ratio > 0",
        "strict_comparable": "performance gate and robust_score > 0",
        "short_period_note": "robust_score is diagnostic when robust_score_comparable is false",
    },
    "outputs": [
        "backtest_metrics.csv",
        "backtest_registry.json",
        "family_composite_vs_screen.csv",
        "family_composite_vs_screen_total.csv",
        "incremental_effects.csv",
        "incremental_effects_total.csv",
        "performance_top_curves.csv",
        "performance_ratios.csv",
        "selection_manifest.csv",
        "run_manifest.json",
    ],
}
SELECTION_MANIFEST.to_csv(EXPORT_DIR / "selection_manifest.csv", index=False)
with (EXPORT_DIR / "run_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(run_manifest, handle, ensure_ascii=False, indent=2)

print(f"结果目录：{EXPORT_DIR}")
display(
    family_comparison.loc[
        family_comparison["period_id"].eq("total"),
        [
            "family",
            "active_cagr_new",
            "active_cagr_screen",
            "delta_active_cagr",
            "top_information_ratio_new",
            "top_information_ratio_screen",
            "delta_top_information_ratio",
            "robust_score_new",
            "robust_score_screen",
            "strict_comparable_improvement",
        ],
    ].sort_values("delta_active_cagr", ascending=False)
)
display(
    incremental_effects.loc[
        incremental_effects["period_id"].eq("total"),
        [
            "family",
            "role",
            "variable",
            "dimension",
            "delta_active_cagr",
            "delta_top_worst_cagr",
            "delta_top_information_ratio",
            "delta_robust_score",
            "incremental_perf_improved",
            "incremental_risk_not_worse",
        ],
    ].sort_values(
        ["family", "delta_active_cagr"], ascending=[True, False]
    )
)


## 5. 运行后的判读顺序

1. 先看 family_composite_vs_screen_total.csv，确认新 composite 相对 screen baseline 的 total 方向。
2. 再看 family_composite_vs_screen.csv 的各 period 行，确认改善不是由单一时期或短样本造成。
3. 再看 incremental_effects_total.csv 与 incremental_effects.csv：只有在收益指标整体改善且风险没有恶化时，才把单腿列为“可加入候选”。
4. 最后回到 backtest_metrics.csv 和 backtest_registry.json 核对完整 composition、period、benchmark、observation_count 与 provenance。

本 notebook 使用的是附件报告中的 Top12 候选子集；运行结果不能被解释为完整变量宇宙的无偏筛选。